# 🔬 MedVLM: Dual-Purpose Ophthalmology Benchmark (Anatomical Recognition & Diagnosis)
### Full Evaluation of InternVL2-4B on OCT (OIMHS) and Color Fundus Photography (REFUGE, ORIGA, G1020, IDRiD)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaJinWakeUp/MedVLM/blob/main/colab_full_dataset_inference.ipynb)

---

### 🎯 Scope & Modality Mapping
In the LMOD benchmark, only modalities with **both** Anatomical Recognition and Clinical Diagnosis annotations are evaluated here:

| Modality | Source Dataset(s) | Sample Count | Anatomical Recognition | Diagnosis Analysis |
|:---|:---|:---:|:---:|:---|
| **OCT** | **OIMHS** | **3,859** | irc, retina, choroid, mh | **Macular Hole Staging** (Stages 1–4) |
| **Color Fundus (CFP)** | **REFUGE, ORIGA, G1020, IDRiD** | **3,386** | Optic Disc, Optic Cup, Fovea | **Glaucoma Detection** (REFUGE, ORIGA, G1020) |

---

### ⚙️ Pipeline Highlights
- 🗂️ **Google Drive Mount & Local Colab SSD Extraction**: Reads zip archives from Google Drive (`Datasets/LMOD/`) and unzips them into Colab's high-speed local disk (`/content/data/LMOD/`). **No files are unzipped inside your Google Drive**.
- 🔄 **Automatic Resume & Two-Way Checkpoint Sync**: Whenever Colab is restarted, existing checkpoint results in `Datasets/LMOD/results/` in your Google Drive are **automatically detected and loaded into Colab**, so inference picks up seamlessly from where you left off without repeating completed samples!
- 📊 **Per-Dataset & Per-Modality Performance Reporting**: Computes detailed metrics (F1, Accuracy, Sensitivity, Specificity, Quadratic Weighted Kappa) broken down by dataset source and overall.
- 💾 **Safe Auto-Backup to Google Drive**: Predictions, reports, and plots are continuously synced to `MyDrive/Datasets/LMOD/results/`.

> ⚠️ **Runtime Requirement**: Go to **Runtime → Change runtime type → T4 GPU (or A100/V100)** before running.


## Step 1: Install Dependencies & Check GPU Environment

We install `transformers` (pinned `<4.46.0` for InternVL compatibility), `accelerate`, `torchvision`, `scikit-learn`, `seaborn`, and `tqdm`.


In [ ]:
%%capture
!pip install "transformers>=4.37.0,<4.46.0" accelerate torchvision scikit-learn seaborn pandas tqdm matplotlib Pillow

print("✅ Dependencies successfully installed!")


In [ ]:
import os
import re
import sys
import glob
import json
import time
import shutil
import zipfile
import tarfile
import textwrap
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, cohen_kappa_score
)

warnings.filterwarnings('ignore')

# ─── GPU Diagnostics ──────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("=" * 65)
print(f"PyTorch Version : {torch.__version__}")
print(f"Device Selected : {device.upper()}")
if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Model       : {gpu_name}")
    print(f"VRAM Available  : {gpu_mem:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Please change runtime type to GPU.")
print("=" * 65)


## Step 2: Mount Google Drive, Restore Checkpoints & Unzip Datasets to Local SSD

We mount your Google Drive at `/content/drive`.
1. **Auto-Restore Existing Checkpoints**: If you previously ran inference, existing results from Google Drive (`Datasets/LMOD/results/`) are automatically restored into Colab's local working directory.
2. **Fast Local Unzipping**: Zip files are extracted directly into Colab's temporary NVMe SSD (`/content/data/LMOD`), leaving Google Drive read-only.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PATH & DATASET CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

# Path to uploaded dataset folder in Google Drive
DRIVE_DATASET_DIR = '/content/drive/MyDrive/Datasets/LMOD'
if not os.path.exists(DRIVE_DATASET_DIR):
    alt_drive_dir = '/content/drive/My Drive/Datasets/LMOD'
    if os.path.exists(alt_drive_dir):
        DRIVE_DATASET_DIR = alt_drive_dir

# Target local working directories on Colab SSD
LOCAL_DATA_ROOT  = '/content/data/LMOD'
RESULTS_DIR      = '/content/data/results'
DRIVE_BACKUP_DIR = os.path.join(DRIVE_DATASET_DIR, 'results')

os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# Select the dual-purpose dataset archives (Anatomy + Diagnosis):
TARGET_ZIPS = ['OIMHS.zip', 'REFUGE.zip', 'ORIGA.zip', 'G1020.zip', 'IDRiD.zip']

print(f"Source Drive Path    : {DRIVE_DATASET_DIR}")
print(f"Drive Checkpoint Dir : {DRIVE_BACKUP_DIR}")
print(f"Local Colab Target   : {LOCAL_DATA_ROOT}")
print(f"Local Results Dir    : {RESULTS_DIR}")
print(f"Target Datasets      : {TARGET_ZIPS}")

# ─── Automatic Checkpoint Restore from Google Drive ──────────────────────────
if os.path.exists(DRIVE_BACKUP_DIR):
    drive_checkpoints = [f for f in os.listdir(DRIVE_BACKUP_DIR) if f.endswith('.json') or f.endswith('.csv')]
    if drive_checkpoints:
        print(f"\n🔄 Found {len(drive_checkpoints)} previous checkpoint file(s) in Google Drive:")
        for cp in drive_checkpoints:
            src = os.path.join(DRIVE_BACKUP_DIR, cp)
            dst = os.path.join(RESULTS_DIR, cp)
            shutil.copy2(src, dst)
            size_kb = os.path.getsize(src) / 1024
            print(f"  -> Restored {cp:35s} ({size_kb:.1f} KB) to Colab local disk")
        print("✅ Previous progress successfully restored! Inference will resume from where you left off.")
    else:
        print("\nℹ️ No previous checkpoints found in Google Drive results folder. Starting fresh.")


In [ ]:
def extract_to_colab_local(drive_dir, local_target, selected_zips):
    """
    Extracts specified zip files from Google Drive to local Colab SSD (/content/data/LMOD).
    """
    if not os.path.exists(drive_dir):
        raise FileNotFoundError(f"❌ Google Drive folder not found at: {drive_dir}")
            
    total, used, free = shutil.disk_usage('/content')
    print(f"💾 Colab Local Disk: {free / (1024**3):.1f} GB free (out of {total / (1024**3):.1f} GB total)")
    
    available_files = os.listdir(drive_dir)
    zips_to_extract = [f for f in selected_zips if f in available_files]
    print(f"\n🔍 Found {len(zips_to_extract)} dataset zip archives in Drive:")
    
    for zname in zips_to_extract:
        zip_path = os.path.join(drive_dir, zname)
        folder_name = os.path.splitext(zname)[0]
        local_folder = os.path.join(local_target, folder_name)
        
        if os.path.exists(local_folder) and os.listdir(local_folder):
            print(f"  ⏩ {zname:14s} already extracted at {local_folder}. Skipping.")
            continue
            
        size_mb = os.path.getsize(zip_path) / (1024**2)
        print(f"  📦 Unzipping {zname:14s} ({size_mb:.1f} MB) -> Colab local SSD ...")
        t0 = time.time()
        
        cmd = f'unzip -q -o "{zip_path}" -d "{local_target}"'
        res = os.system(cmd)
        if res != 0:
            print(f"     Falling back to python zipfile for {zname}...")
            with zipfile.ZipFile(zip_path, 'r') as zf:
                zf.extractall(local_target)
                
        elapsed = time.time() - t0
        print(f"     ✅ Extracted {zname} in {elapsed:.1f}s")
        
    print(f"\n🎉 Datasets ready on Colab local SSD at: {local_target}")

# Execute extraction
extract_to_colab_local(DRIVE_DATASET_DIR, LOCAL_DATA_ROOT, TARGET_ZIPS)


## Step 3: Dataset Discovery & Indexing Across Dual-Purpose Modalities

We index all sample directories containing `information.json` and group them by modality (**OCT** vs. **Color Fundus**) and dataset source (**OIMHS, REFUGE, ORIGA, G1020, IDRiD**).


In [ ]:
def index_dual_purpose_datasets(root_dir):
    """
    Crawls local root_dir for all sample directories containing information.json.
    """
    oct_samples = []
    cfp_samples = []
    
    info_files = glob.glob(os.path.join(root_dir, '**', 'information.json'), recursive=True)
    print(f"Found {len(info_files)} total 'information.json' files across extracted folders.")
    
    for info_path in tqdm(info_files, desc="Indexing dataset metadata"):
        sample_dir = os.path.dirname(info_path)
        sample_id  = os.path.basename(sample_dir)
        
        try:
            with open(info_path, 'r', encoding='utf-8') as f:
                meta = json.load(f)
        except Exception:
            continue
            
        img_type = str(meta.get('image_type', '')).lower()
        meta_dict = meta.get('metadata', {})
        dataset_id = str(meta_dict.get('dataset_id', '')).lower()
        
        rel_path = os.path.relpath(info_path, root_dir)
        top_folder = rel_path.split(os.sep)[0].upper()
        if 'OIMHS' in top_folder or 'oimhs' in dataset_id:
            source_dataset = 'OIMHS'
        elif 'REFUGE' in top_folder or 'refuge' in dataset_id:
            source_dataset = 'REFUGE'
        elif 'ORIGA' in top_folder or 'origa' in dataset_id:
            source_dataset = 'ORIGA'
        elif 'G1020' in top_folder or 'g1020' in dataset_id:
            source_dataset = 'G1020'
        elif 'IDRID' in top_folder or 'idrid' in dataset_id:
            source_dataset = 'IDRiD'
        else:
            source_dataset = top_folder
            
        annotated_img_path = None
        clean_img_path = None
        
        vis_path = os.path.join(sample_dir, 'visualization.png')
        if os.path.isfile(vis_path):
            clean_img_path = vis_path
            
        annotated_dir = os.path.join(sample_dir, 'annotated')
        if os.path.isdir(annotated_dir):
            for candidate in ['bbox_annotated.png', 'annotated_bounding_box.png', 'annotated.png']:
                cand_path = os.path.join(annotated_dir, candidate)
                if os.path.isfile(cand_path):
                    annotated_img_path = cand_path
                    break
            if not annotated_img_path:
                pngs = glob.glob(os.path.join(annotated_dir, '*.png'))
                if pngs:
                    annotated_img_path = pngs[0]
                    
        if not annotated_img_path:
            annotated_img_path = clean_img_path
        if not clean_img_path:
            clean_img_path = annotated_img_path
            
        if not clean_img_path or not os.path.exists(clean_img_path):
            continue
            
        bboxes = meta.get('annotations', {}).get('bounding_boxes', [])
        
        if source_dataset == 'OIMHS' or 'oct' in img_type:
            mh_stage = meta_dict.get('stage_of_macular_hole_decision')
            oct_samples.append({
                'id': sample_id,
                'source_dataset': source_dataset,
                'dir': sample_dir,
                'info_path': info_path,
                'clean_image': clean_img_path,
                'annotated_image': annotated_img_path,
                'meta': meta,
                'mh_stage': int(mh_stage) if mh_stage is not None else None,
                'bboxes': bboxes,
                'modality': 'OCT'
            })
        else:
            g_label = meta_dict.get('glaucoma_label')
            if g_label is not None:
                g_str = str(g_label).strip()
                if g_str.lower() in ['glaucoma', '1', 'true', '1.0']:
                    g_label_norm = 'Glaucoma'
                else:
                    g_label_norm = 'Non-Glaucoma'
            else:
                g_label_norm = None
                
            cfp_samples.append({
                'id': sample_id,
                'source_dataset': source_dataset,
                'dir': sample_dir,
                'info_path': info_path,
                'clean_image': clean_img_path,
                'annotated_image': annotated_img_path,
                'meta': meta,
                'glaucoma_label': g_label_norm,
                'bboxes': bboxes,
                'modality': 'CFP'
            })
            
    return oct_samples, cfp_samples

oct_dataset, cfp_dataset = index_dual_purpose_datasets(LOCAL_DATA_ROOT)

print("=" * 70)
print("📊 DUAL-PURPOSE DATASET BREAKDOWN:")
print("=" * 70)
print(f"1. OCT MODALITY (Total: {len(oct_dataset)} scans):")
print(f"   • OIMHS: {len(oct_dataset)} images (MH Stages 1-4 & Retinal layers)")

print(f"\n2. COLOR FUNDUS MODALITY (Total: {len(cfp_dataset)} photos):")
cfp_by_source = Counter([s['source_dataset'] for s in cfp_dataset])
for src, cnt in cfp_by_source.items():
    g_cnt = sum(1 for s in cfp_dataset if s['source_dataset'] == src and s['glaucoma_label'] is not None)
    b_cnt = sum(len(s['bboxes']) for s in cfp_dataset if s['source_dataset'] == src)
    print(f"   • {src:10s}: {cnt:4d} images | {g_cnt:4d} Glaucoma labels | {b_cnt:4d} Bounding Boxes")
print("=" * 70)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# INFERENCE SETTINGS
# ─────────────────────────────────────────────────────────────────────────────

# Set to None for FULL dataset evaluation.
# Or set an integer (e.g. 100) for a rapid testing run.
MAX_OCT_SAMPLES = None    # e.g. None or 100
MAX_CFP_SAMPLES = None    # e.g. None or 100

CHECKPOINT_INTERVAL = 50

run_oct_samples = oct_dataset[:MAX_OCT_SAMPLES] if MAX_OCT_SAMPLES else oct_dataset
run_cfp_samples = cfp_dataset[:MAX_CFP_SAMPLES] if MAX_CFP_SAMPLES else cfp_dataset

print(f"Execution Plan:")
print(f"  -> OCT Samples (Anatomy & MH Staging)    : {len(run_oct_samples)}")
print(f"  -> CFP Samples (Anatomy & Glaucoma Diag) : {len(run_cfp_samples)}")


## Step 4: Load InternVL2-4B Model & Tokenizer

We load `OpenGVLab/InternVL2-4B` in `torch.bfloat16` precision.


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def preprocess_image(pil_image, input_size=448):
    """Resize and normalize PIL image for InternVL input."""
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])
    return transform(pil_image).unsqueeze(0).to(torch.bfloat16).to(device)


MODEL_NAME = 'OpenGVLab/InternVL2-4B'
print(f"Loading {MODEL_NAME} from HuggingFace Hub...")

try:
    model = AutoModel.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        use_flash_attn=True,
        trust_remote_code=True,
    ).eval().to(device)
except Exception:
    print("FlashAttention not available, loading standard attention...")
    model = AutoModel.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        use_flash_attn=False,
        trust_remote_code=True,
    ).eval().to(device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=False)
generation_config = dict(max_new_tokens=512, do_sample=False)

print(f"\n✅ {MODEL_NAME} loaded on {device.upper()}!")


## Step 5: Task 1 Full Inference — Anatomical Recognition (OCT & CFP)

**Clinical Goal**: Identify anatomical structures from expert bounding boxes:
- **OCT Scans (OIMHS)**: `irc`, `retina`, `choroid`, `mh`
- **Color Fundus (REFUGE, ORIGA, G1020, IDRiD)**: `optic disc`, `optic cup`, `fovea`, `disc`


In [ ]:
def build_anatomy_prompt(bboxes, modality='OCT'):
    """Constructs prompt tailored for OCT or Color Fundus anatomical recognition."""
    region_ids = [str(bb.get('annotation_ID', idx+1)) for idx, bb in enumerate(bboxes)]
    if modality == 'OCT':
        options = "irc, retina, choroid, mh"
        desc = "ophthalmology OCT image"
    else:
        options = "optic disc, optic cup, fovea, disc"
        desc = "ophthalmology color fundus image"
        
    return (
        f"This is an {desc}. "
        f"Please identify the type of each labeled bounding box in this image. "
        f"Options can be: {options}. "
        "Please just follow the format: "
        + "; ".join([f"Region ID: {rid}; Type: <answer>" for rid in region_ids])
    )

def parse_anatomy_response(response):
    """Extract {region_id: predicted_type} mapping."""
    predictions = {}
    pairs = re.findall(
        r'region\s+id[:\s]+(\d+)[^a-z]*type[:\s]+([a-z\s]+?)(?:;|$|\n)',
        response, re.IGNORECASE
    )
    for rid, rtype in pairs:
        clean_t = rtype.strip().lower().rstrip('.,;')
        predictions[str(rid)] = clean_t
    return predictions


TASK1_SAVE_PATH = os.path.join(RESULTS_DIR, 'task1_anatomy_results.json')
DRIVE_TASK1_PATH = os.path.join(DRIVE_BACKUP_DIR, 'task1_anatomy_results.json')
task1_results = []
processed_ids = set()

# 1. Check local SSD first, then check Google Drive if local is missing
if not os.path.exists(TASK1_SAVE_PATH) and os.path.exists(DRIVE_TASK1_PATH):
    try:
        shutil.copy2(DRIVE_TASK1_PATH, TASK1_SAVE_PATH)
        print("🔄 Copied existing Task 1 checkpoint from Google Drive to local disk.")
    except Exception as e:
        print(f"⚠️ Warning copying Task 1 checkpoint from Drive: {e}")

if os.path.exists(TASK1_SAVE_PATH):
    try:
        with open(TASK1_SAVE_PATH, 'r', encoding='utf-8') as f:
            task1_results = json.load(f)
        processed_ids = {r['sample_id'] for r in task1_results}
        print(f"🔄 Resuming Task 1: Found {len(task1_results)} previously processed samples.")
    except Exception as e:
        print(f"⚠️ Checkpoint load issue: {e}")

all_anatomy_samples = run_oct_samples + run_cfp_samples
remaining_samples = [s for s in all_anatomy_samples if f"{s['source_dataset']}_{s['id']}" not in processed_ids]

print(f"Total Task 1 Target Samples    : {len(all_anatomy_samples)}")
print(f"Already Completed in Drive     : {len(processed_ids)}")
print(f"Remaining Samples to Evaluate  : {len(remaining_samples)}")

if not remaining_samples:
    print("\n🎉 ALL Task 1 Anatomical Recognition samples are already completed! Skipping inference and proceeding to evaluation.")
else:
    print(f"\n🚀 Starting Task 1 Inference on {len(remaining_samples)} remaining images...")
    t0 = time.time()
    
    for idx, sample in enumerate(tqdm(remaining_samples, desc="Task 1: Anatomical Recognition")):
        s_id = f"{sample['source_dataset']}_{sample['id']}"
        bboxes = sample.get('bboxes', [])
        if not bboxes:
            continue
            
        gt_mapping = {str(bb.get('annotation_ID', i+1)): str(bb.get('region_type', '')).strip().lower() for i, bb in enumerate(bboxes)}
        
        try:
            img = Image.open(sample['annotated_image']).convert('RGB')
            pixel_values = preprocess_image(img)
            prompt = f"<image>\n{build_anatomy_prompt(bboxes, sample['modality'])}"
            
            with torch.no_grad():
                response = model.chat(tokenizer, pixel_values, prompt, generation_config)
                
            preds = parse_anatomy_response(response)
            
            correct_count = sum(preds.get(rid, '') == rtype for rid, rtype in gt_mapping.items())
            total_count   = len(gt_mapping)
            acc           = correct_count / total_count if total_count > 0 else 0.0
            
            record = {
                'sample_id': s_id,
                'dataset': sample['source_dataset'],
                'modality': sample['modality'],
                'ground_truth': gt_mapping,
                'predictions': preds,
                'accuracy': acc,
                'correct_count': correct_count,
                'total_count': total_count,
                'response': response
            }
            task1_results.append(record)
            
        except Exception as e:
            pass
            
        # Checkpoint periodically to local disk & Google Drive
        if (idx + 1) % CHECKPOINT_INTERVAL == 0:
            with open(TASK1_SAVE_PATH, 'w', encoding='utf-8') as f:
                json.dump(task1_results, f, indent=2)
            try:
                shutil.copy2(TASK1_SAVE_PATH, DRIVE_BACKUP_DIR)
            except Exception:
                pass

    # Final save
    with open(TASK1_SAVE_PATH, 'w', encoding='utf-8') as f:
        json.dump(task1_results, f, indent=2)

    try:
        shutil.copy2(TASK1_SAVE_PATH, DRIVE_BACKUP_DIR)
    except Exception:
        pass

    elapsed_t1 = time.time() - t0
    print(f"\n✅ Task 1 Completed in {elapsed_t1/60:.2f} mins! Total evaluated: {len(task1_results)}")


In [ ]:
# ─── Task 1 Evaluation Breakdown by Dataset & Modality ─────────────────────────
df_t1 = pd.DataFrame(task1_results)

print("=" * 70)
print("📊 TASK 1: ANATOMICAL RECOGNITION PERFORMANCE SUMMARY")
print("=" * 70)

for modality in ['OCT', 'CFP']:
    df_mod = df_t1[df_t1['modality'] == modality]
    if df_mod.empty:
        continue
        
    print(f"\n▶ Modality: {modality} ({len(df_mod)} samples)")
    for dset in df_mod['dataset'].unique():
        df_dset = df_mod[df_mod['dataset'] == dset]
        tot_boxes = df_dset['total_count'].sum()
        cor_boxes = df_dset['correct_count'].sum()
        acc = cor_boxes / tot_boxes if tot_boxes > 0 else 0.0
        print(f"  • {dset:10s} : Box Accuracy = {acc:.2%} ({cor_boxes}/{tot_boxes} boxes across {len(df_dset)} images)")
        
    overall_mod_cor = df_mod['correct_count'].sum()
    overall_mod_tot = df_mod['total_count'].sum()
    print(f"  ★ {modality} Overall Box Accuracy: {overall_mod_cor/overall_mod_tot:.2%}")

# Overall OCT Macro F1 (Comparable to Paper)
oct_true, oct_pred = [], []
for r in [r for r in task1_results if r['modality'] == 'OCT']:
    for rid, true_t in r['ground_truth'].items():
        oct_true.append(true_t)
        oct_pred.append(r['predictions'].get(rid, 'unknown'))
        
oct_macro_f1 = precision_recall_fscore_support(oct_true, oct_pred, average='macro', zero_division=0)[2] if oct_true else 0.0
print(f"\nOCT (OIMHS) Macro F1 : {oct_macro_f1:.4f} (Paper InternVL-2B: 0.4807 | Specialist NN: 0.9840)")
print("=" * 70)


## Step 6: Task 2 Full Inference — Diagnosis Analysis (OCT & CFP)

**Clinical Goals**:
1. **OCT Diagnosis**: Macular Hole Staging (Stages 1–4 on OIMHS scans).
2. **CFP Diagnosis**: Glaucoma Detection (Glaucoma vs. Non-Glaucoma on REFUGE, ORIGA, G1020).


In [ ]:
# ─── 1. OCT Diagnosis: Macular Hole Staging ──────────────────────────────────
MH_PROMPT = (
    "This is an ophthalmology OCT image. "
    "Based on the image, please tell me the stage of macular hole decision. "
    "Then, give a detailed justification and explanation for your answer. "
    "Follow the format: Stage: <AN INTEGER>; Justification: <EXPLANATION>."
)

def parse_mh_stage(response):
    match = re.search(r'stage[:\s]+([1-4])', response, re.IGNORECASE)
    if match:
        return int(match.group(1))
    match = re.search(r'\b([1-4])\b', response)
    return int(match.group(1)) if match else None


# ─── 2. CFP Diagnosis: Glaucoma Detection ────────────────────────────────────
GLAUCOMA_PROMPT = (
    "This is a color fundus image of type Fundus RGB Images. "
    "Based on the image, please tell me whether this image contains glaucoma, "
    "then give detailed justifications. "
    "Follow the format: GLAUCOMA / NON-GLAUCOMA; Explanation: <JUSTIFICATION>."
)

def parse_glaucoma(response):
    r = response.upper()
    if 'NON-GLAUCOMA' in r or 'NON GLAUCOMA' in r or 'NOT GLAUCOMA' in r:
        return 'Non-Glaucoma'
    elif 'GLAUCOMA' in r:
        return 'Glaucoma'
    return 'Unknown'


TASK2_SAVE_PATH = os.path.join(RESULTS_DIR, 'task2_diagnosis_results.json')
DRIVE_TASK2_PATH = os.path.join(DRIVE_BACKUP_DIR, 'task2_diagnosis_results.json')
task2_results = []
processed_diag_ids = set()

# Check local SSD first, then check Google Drive if local is missing
if not os.path.exists(TASK2_SAVE_PATH) and os.path.exists(DRIVE_TASK2_PATH):
    try:
        shutil.copy2(DRIVE_TASK2_PATH, TASK2_SAVE_PATH)
        print("🔄 Copied existing Diagnosis checkpoint from Google Drive to local disk.")
    except Exception as e:
        print(f"⚠️ Warning copying Diagnosis checkpoint from Drive: {e}")

if os.path.exists(TASK2_SAVE_PATH):
    try:
        with open(TASK2_SAVE_PATH, 'r', encoding='utf-8') as f:
            task2_results = json.load(f)
        processed_diag_ids = {r['sample_id'] for r in task2_results}
        print(f"🔄 Resuming Diagnosis: Found {len(task2_results)} previously processed samples.")
    except Exception as e:
        print(f"⚠️ Checkpoint load issue: {e}")

diag_samples = (
    [s for s in run_oct_samples if s['mh_stage'] is not None] +
    [s for s in run_cfp_samples if s['glaucoma_label'] is not None]
)
remaining_diag_samples = [s for s in diag_samples if f"{s['source_dataset']}_{s['id']}" not in processed_diag_ids]

print(f"Total Diagnosis Target Samples : {len(diag_samples)}")
print(f"Already Completed in Drive     : {len(processed_diag_ids)}")
print(f"Remaining Samples to Evaluate  : {len(remaining_diag_samples)}")

if not remaining_diag_samples:
    print("\n🎉 ALL Diagnosis samples are already completed! Skipping inference and proceeding to evaluation.")
else:
    print(f"\n🚀 Starting Diagnosis Inference on {len(remaining_diag_samples)} remaining images...")
    t0 = time.time()
    
    for idx, sample in enumerate(tqdm(remaining_diag_samples, desc="Task 2: Diagnosis Analysis")):
        s_id = f"{sample['source_dataset']}_{sample['id']}"
        
        try:
            img = Image.open(sample['clean_image']).convert('RGB')
            pixel_values = preprocess_image(img)
            
            if sample['modality'] == 'OCT':
                prompt = f'<image>\n{MH_PROMPT}'
                with torch.no_grad():
                    response = model.chat(tokenizer, pixel_values, prompt, generation_config)
                pred = parse_mh_stage(response)
                gt   = sample['mh_stage']
                correct = (pred == gt)
                diag_task = 'Macular Hole Staging'
            else:
                prompt = f'<image>\n{GLAUCOMA_PROMPT}'
                with torch.no_grad():
                    response = model.chat(tokenizer, pixel_values, prompt, generation_config)
                pred = parse_glaucoma(response)
                gt   = sample['glaucoma_label']
                correct = (pred == gt)
                diag_task = 'Glaucoma Detection'
                
            record = {
                'sample_id': s_id,
                'dataset': sample['source_dataset'],
                'modality': sample['modality'],
                'diag_task': diag_task,
                'ground_truth': gt,
                'prediction': pred,
                'correct': bool(correct),
                'response': response
            }
            task2_results.append(record)
            
        except Exception as e:
            pass
            
        if (idx + 1) % CHECKPOINT_INTERVAL == 0:
            with open(TASK2_SAVE_PATH, 'w', encoding='utf-8') as f:
                json.dump(task2_results, f, indent=2)
            try:
                shutil.copy2(TASK2_SAVE_PATH, DRIVE_BACKUP_DIR)
            except Exception:
                pass

    with open(TASK2_SAVE_PATH, 'w', encoding='utf-8') as f:
        json.dump(task2_results, f, indent=2)

    try:
        shutil.copy2(TASK2_SAVE_PATH, DRIVE_BACKUP_DIR)
    except Exception:
        pass

    elapsed_t2 = time.time() - t0
    print(f"\n✅ Diagnosis Task Completed in {elapsed_t2/60:.2f} mins! Total evaluated: {len(task2_results)}")


In [ ]:
# ─── Task 2 Metrics: Detailed Breakdown ───────────────────────────────────────
df_diag = pd.DataFrame(task2_results)

print("=" * 70)
print("📊 DIAGNOSIS ANALYSIS: EVALUATION REPORT")
print("=" * 70)

# 1. OCT: Macular Hole Staging
df_oct_diag = df_diag[df_diag['modality'] == 'OCT'] if not df_diag.empty else pd.DataFrame()
if not df_oct_diag.empty:
    valid_oct = df_oct_diag.dropna(subset=['prediction', 'ground_truth'])
    y_t_oct = [int(v) for v in valid_oct['ground_truth']]
    y_p_oct = [int(v) for v in valid_oct['prediction']]
    acc_oct = accuracy_score(y_t_oct, y_p_oct)
    qwk_oct = cohen_kappa_score(y_t_oct, y_p_oct, weights='quadratic')
    mae_oct = np.mean(np.abs(np.array(y_t_oct) - np.array(y_p_oct)))
    print("▶ 1. OCT MACULAR HOLE STAGING (OIMHS):")
    print(f"  • Evaluated Samples        : {len(valid_oct)}")
    print(f"  • Overall Accuracy         : {acc_oct:.2%} (Paper InternVL-2B: 30.3% | Specialist: 98.2%)")
    print(f"  • Quadratic Weighted Kappa : {qwk_oct:.4f}")
    print(f"  • Mean Absolute Error (MAE): {mae_oct:.2f} stages\n")

# 2. CFP: Glaucoma Detection Breakdown
df_cfp_diag = df_diag[df_diag['modality'] == 'CFP'] if not df_diag.empty else pd.DataFrame()
if not df_cfp_diag.empty:
    print("▶ 2. COLOR FUNDUS GLAUCOMA DETECTION (Per Dataset Breakdown):")
    for dset in df_cfp_diag['dataset'].unique():
        df_d = df_cfp_diag[df_cfp_diag['dataset'] == dset]
        clean_d = df_d[df_d['prediction'].isin(['Glaucoma', 'Non-Glaucoma'])]
        if clean_d.empty:
            continue
        yt = clean_d['ground_truth'].tolist()
        yp = clean_d['prediction'].tolist()
        acc = accuracy_score(yt, yp)
        cm = confusion_matrix(yt, yp, labels=['Glaucoma', 'Non-Glaucoma'])
        tp, fn, fp, tn = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        print(f"  • {dset:8s} ({len(clean_d)} images): Acc = {acc:.2%} | Sens = {sens:.2%} | Spec = {spec:.2%}")
        
    clean_all_cfp = df_cfp_diag[df_cfp_diag['prediction'].isin(['Glaucoma', 'Non-Glaucoma'])]
    yt_all = clean_all_cfp['ground_truth'].tolist()
    yp_all = clean_all_cfp['prediction'].tolist()
    overall_g_acc = accuracy_score(yt_all, yp_all)
    cm_all = confusion_matrix(yt_all, yp_all, labels=['Glaucoma', 'Non-Glaucoma'])
    tp, fn, fp, tn = cm_all[0][0], cm_all[0][1], cm_all[1][0], cm_all[1][1]
    overall_sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    overall_spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    print(f"\n  ★ ALL CFP Glaucoma Combined ({len(clean_all_cfp)} images):")
    print(f"     Accuracy    : {overall_g_acc:.2%}")
    print(f"     Sensitivity : {overall_sens:.2%}")
    print(f"     Specificity : {overall_spec:.2%}")
print("=" * 70)


## Step 7: Comprehensive Visualizations & Diagnostic Dashboard

We plot confusion matrices, dataset comparison breakdowns, and benchmark baselines.


In [ ]:
# ─── Multi-Panel Confusion Matrices & Breakdown ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle('MedVLM Benchmark: Dual-Purpose Modalities Evaluation Dashboard', fontsize=16, fontweight='bold', y=1.02)

# Panel 1: OCT Anatomy (OIMHS)
if oct_true:
    cm_oct = confusion_matrix(oct_true, oct_pred, labels=['irc', 'retina', 'choroid', 'mh'])
    sns.heatmap(cm_oct, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['irc', 'retina', 'choroid', 'mh'], yticklabels=['irc', 'retina', 'choroid', 'mh'], cbar=False)
    axes[0].set_title(f'OCT Anatomical Recognition (OIMHS)\n(Macro F1: {oct_macro_f1:.3f})', fontweight='bold')
    axes[0].set_xlabel('Predicted Region')
    axes[0].set_ylabel('True Region')

# Panel 2: OCT Diagnosis (MH Staging)
if not df_oct_diag.empty:
    cm_mh = confusion_matrix(y_t_oct, y_p_oct, labels=[1, 2, 3, 4])
    sns.heatmap(cm_mh, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
                xticklabels=['S1', 'S2', 'S3', 'S4'], yticklabels=['S1', 'S2', 'S3', 'S4'], cbar=False)
    axes[1].set_title(f'OCT Macular Hole Staging (OIMHS)\n(Acc: {acc_oct:.1%}, QWK: {qwk_oct:.3f})', fontweight='bold')
    axes[1].set_xlabel('Predicted Stage')
    axes[1].set_ylabel('True Stage')

# Panel 3: CFP Diagnosis (Glaucoma Detection across REFUGE, ORIGA, G1020)
if not df_cfp_diag.empty:
    sns.heatmap(cm_all, annot=True, fmt='d', cmap='Greens', ax=axes[2],
                xticklabels=['Glaucoma', 'Non-Glaucoma'], yticklabels=['Glaucoma', 'Non-Glaucoma'], cbar=False)
    axes[2].set_title(f'CFP Glaucoma Diagnosis (REFUGE, ORIGA, G1020)\n(Acc: {overall_g_acc:.1%}, Sens: {overall_sens:.1%}, Spec: {overall_spec:.1%})', fontweight='bold')
    axes[2].set_xlabel('Predicted Diagnosis')
    axes[2].set_ylabel('True Diagnosis')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'dual_purpose_evaluation_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─── Dataset Breakdown Comparison Bar Chart ──────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))

chart_labels = []
chart_scores = []
chart_colors = []

if oct_true:
    chart_labels.append('OCT Anatomy\n(OIMHS)')
    chart_scores.append(oct_macro_f1)
    chart_colors.append('#3498db')

if not df_oct_diag.empty:
    chart_labels.append('OCT MH Staging\n(OIMHS)')
    chart_scores.append(acc_oct)
    chart_colors.append('#e67e22')

if not df_cfp_diag.empty:
    for dset in ['REFUGE', 'ORIGA', 'G1020']:
        df_d = df_cfp_diag[df_cfp_diag['dataset'] == dset]
        clean_d = df_d[df_d['prediction'].isin(['Glaucoma', 'Non-Glaucoma'])]
        if not clean_d.empty:
            chart_labels.append(f'Glaucoma Diag\n({dset})')
            chart_scores.append(accuracy_score(clean_d['ground_truth'], clean_d['prediction']))
            chart_colors.append('#2ecc71')

if chart_labels:
    bars = ax.bar(chart_labels, chart_scores, color=chart_colors, edgecolor='black', alpha=0.85, width=0.5)
    ax.set_ylabel('Performance Score (Accuracy / F1)', fontsize=11, fontweight='bold')
    ax.set_title('InternVL2-4B Performance on Dual-Purpose Datasets (OCT & CFP)', fontsize=14, fontweight='bold', pad=15)
    ax.set_ylim(0, 1.15)
    ax.grid(axis='y', alpha=0.3)

    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.1%}',
                    xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 4), textcoords="offset points",
                    ha='center', va='bottom', fontsize=9.5, fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'dual_purpose_dataset_breakdown.png'), dpi=150, bbox_inches='tight')
    plt.show()


## Step 8: Export Comprehensive Results to Google Drive

We save prediction tables (CSV), structured evaluation reports (JSON), and summary figures to `MyDrive/Datasets/LMOD/results/`.


In [ ]:
# ─── Export CSV DataFrames ───────────────────────────────────────────────────
df_t1 = pd.DataFrame(task1_results)
df_t2 = pd.DataFrame(task2_results)

csv1_path = os.path.join(RESULTS_DIR, 'anatomical_recognition_all_predictions.csv')
csv2_path = os.path.join(RESULTS_DIR, 'diagnosis_all_predictions.csv')

df_t1.to_csv(csv1_path, index=False)
df_t2.to_csv(csv2_path, index=False)

# ─── Export Summary JSON ──────────────────────────────────────────────────────
summary_report = {
    'model': MODEL_NAME,
    'timestamp': time.strftime("%Y-%m-%d %H:%M:%S"),
    'oct_oimhs': {
        'anatomical_recognition_macro_f1': float(oct_macro_f1) if oct_true else None,
        'mh_staging_accuracy': float(acc_oct) if not df_oct_diag.empty else None,
        'mh_staging_qwk': float(qwk_oct) if not df_oct_diag.empty else None
    },
    'cfp_glaucoma': {
        'combined_accuracy': float(overall_g_acc) if not df_cfp_diag.empty else None,
        'combined_sensitivity': float(overall_sens) if not df_cfp_diag.empty else None,
        'combined_specificity': float(overall_spec) if not df_cfp_diag.empty else None
    }
}

summary_json_path = os.path.join(RESULTS_DIR, 'dual_purpose_summary_report.json')
with open(summary_json_path, 'w', encoding='utf-8') as f:
    json.dump(summary_report, f, indent=2)

# ─── Sync All Results to Google Drive ─────────────────────────────────────────
print(f"📦 Syncing results to Google Drive folder: {DRIVE_BACKUP_DIR} ...")
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

for fname in os.listdir(RESULTS_DIR):
    src_file = os.path.join(RESULTS_DIR, fname)
    dst_file = os.path.join(DRIVE_BACKUP_DIR, fname)
    if os.path.isfile(src_file):
        shutil.copy2(src_file, dst_file)
        print(f"  -> Synced {fname} to Google Drive")

print("\n" + "=" * 65)
print("🎉 DUAL-PURPOSE DATASET BENCHMARK COMPLETED SUCCESSFULLY!")
print(f"📁 Results saved at: {DRIVE_BACKUP_DIR}")
print("=" * 65)
